# Week 12: RoBERTa Fine-Tuning
Task 4 | Aigerim Dairanbek

Fine-tune (cardiffnlp/twitter-roberta-base-sentiment-latest) on IMDb,
then evaluate zero-shot on tweets using the same threshold sweep as the baseline.


**0. Setup**

In [4]:
!git clone https://github.com/imrushka/ML_group_project
%cd ML_group_project

Cloning into 'ML_group_project'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 31 (delta 12), reused 28 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 16.35 KiB | 8.17 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/ML_group_project


In [5]:
%pip install -q datasets emoji pyarrow joblib
%pip install -q transformers accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 18.7 MB/s eta 0:00:00


**1. Data Processing**

In [6]:
from pathlib import Path

processed_files = list(Path('data/processed').glob('imdb_*.csv'))
if len(processed_files) >= 3:
    print(f'Processed data found ({len(processed_files)} files) — skipping pipeline.')
else:
    print('Running data pipeline...')
    %run src/data_collection.py
    %run src/data_cleaning.py

Running data pipeline...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

...

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

/content/ML_group_project/src/data_collection.py:72: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "collected_at": datetime.utcnow().isoformat() + "Z",


In [7]:
!ls src/

baseline_models.py  data_cleaning.py	__pycache__
config.py	    data_collection.py


In [11]:
from google.colab import files
uploaded = files.upload()

Saving fine_tuning.py to fine_tuning (1).py


In [12]:
!mv /content/fine_tuning.py /content/ML_group_project/src/

In [13]:
!ls src/

baseline_models.py  data_cleaning.py	fine_tuning.py
config.py	    data_collection.py	__pycache__


In [18]:
import os
os.chdir('/content/ML_group_project/src')


**2. Run Fine-Tuning**
- Fine-tunes for 3 epochs on 35k IMDb reviews
- Evaluates on IMDb val + test (binary: negative / positive)
- Runs zero-shot threshold sweep [0.55, 0.65, 0.75] on tweets (3-class)
- Saves model → models/roberta_finetuned/
- Saves metrics → logs/finetune_metrics.json


In [19]:
%run fine_tuning.py

Device : cuda
Split sizes:
  train: 35,000 rows | {'positive': 17500, 'negative': 17500}
  val  :  7,500 rows | {'negative': 3750, 'positive': 3750}
  test :  7,500 rows | {'negative': 3750, 'positive': 3750}
  tweet:  1,740 rows | {'negative': 580, 'positive': 580, 'neutral': 580}

Loading tokenizer : cardiffnlp/twitter-roberta-base-sentiment-latest
Loading model     : cardiffnlp/twitter-roberta-base-sentiment-latest


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



Starting fine-tuning on IMDb ...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.256006,0.262198,0.902700,0.902700
2,0.189936,0.266616,0.906400,0.906400
3,0.126484,0.302620,0.907100,0.907000


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Training finished in 13.6 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved → ../models/roberta_finetuned/

── RoBERTa | IMDb val ──
              precision    recall  f1-score   support

    negative       0.92      0.89      0.91      3750
    positive       0.90      0.92      0.91      3750

    accuracy                           0.91      7500
   macro avg       0.91      0.91      0.91      7500
weighted avg       0.91      0.91      0.91      7500


── RoBERTa | IMDb test ──
              precision    recall  f1-score   support

    negative       0.92      0.90      0.91      3750
    positive       0.90      0.92      0.91      3750

    accuracy                           0.91      7500
   macro avg       0.91      0.91      0.91      7500
weighted avg       0.91      0.91      0.91      7500




  RoBERTa — Zero-Shot on Tweets


 threshold=0.55 | Accuracy=0.5529 | Macro-F1=0.4675
              precision    recall  f1-score   support

    negative       0.54      0.82      0.65       580
     neutral       0.56      0.05      0.09       58

---
## 4. Results Summary

| Model | IMDb F1 | Tweet F1 (zero-shot) | Domain Gap |
|---|---|---|---|
| Logistic Regression | ~0.89 | ~0.55 | ~0.34 |
| Linear SVM | ~0.91 | ~0.57 | ~0.34 |
| **RoBERTa** | **0.91** | **0.55** | **0.36** |


**Key finding:** RoBERTa matches SVM on IMDb but does not outperform it
zero-shot on tweets, confirming that model strength alone does not
resolve domain shift. This motivates the Week 13 adaptation step.

---
## Handoff to Week 13 (Domain Adaptation)

| File | Used for |
|---|---|
| models/roberta_finetuned/ | Starting point for domain adaptation |
| logs/finetune_metrics.json | RoBERTa baseline F1 numbers to beat |
| data/processed/tweet_unlabelled_pool.csv | Unlabelled tweets for adaptation |

**Targets for Week 13:**
- Current tweet F1: 0.55 → target ≥ 0.60 after adaptation
- Reduce domain gap from 0.36